<a href="https://colab.research.google.com/github/Sharmi777/IM-Project-Vibration-Comfort-Estimation/blob/main/Vibration_Comfort_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
#For polyphase sampling we import the following from scipy.signals library.
#This function resamples signals at a different frequency without introducing aliasing for higher original sampling frequencies

from scipy.signal import resample_poly

# Anti-aliasing Filter first uses a low pass to remove frequencies higher than the new sampling frequency/2
# to prevent aliasing and then down samples it by removing the excess samples at equal intervals
# Polyphase filtering makes the process faster by splitting the filter cofficients into M parallel branches that
# are evaluated at the same time with rpoper phase difference that is added together to make the filter again

#To find the gcd for the downsampling process we use gcd function of math library
from math import gcd

#For true shuffling and splitting of the data into learning and test datasets we use
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec #For subplots and grid specs

In [ ]:
from google.colab import files

image_files = [
    "scatter_predicted_vs_actual.png",
    "mape_comparison.png",
    "wal_vs_pga_CE16.png",
    "preprocessing_pipeline.png"
]

for filename in image_files:
    try:
        files.download(filename)
    except Exception as e:
        print(f"Error downloading {filename}: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Creating the Linear Regression Model From Scratch by defining a class
class LinearRegression_TRY:
    def __init__(self, learning_rate=0.01, iterations=1000): #initialises the learning rate and number of iterations and assigna weight/intercept to None
        self.lr = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Initialize parameters to zero
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Gradient Descent
        for _ in range(self.iterations):
            # Predict current y = X.weight + bias (this is in the form of matrix equations thus dot product is used)
            y_predicted = np.dot(X, self.weights) + self.bias

            # 2. Calculate gradients (derivatives)
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y)) #derivative of gradient normalised by number of samples
            db = (1 / n_samples) * np.sum(y_predicted - y) #derivative of intercept/bias normalised by number of samples

            # 3. Update weights and bias
            self.weights -= self.lr * dw #descending
            self.bias -= self.lr * db

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

In [ ]:
# We do the pre-processing for the first 20 ground motion data records and their raw acceleration Time series
# 1. Normalisation and Scaling of Data
# 2. Resampling at 50Hz
# 3. Duration Standardisation acc to given convention
# 4. WAL Calculation

EXCEL_FILE   = "Dummy Data Set.xlsx"
TARGET_FS    = 50          # Hz
TARGET_DUR   = 60          # seconds
SCALING      = [1.0, 0.85, 0.65]   # X, Y, Z
BETA_VALUES  = [0.4, 0.8, 1.0, 1.2, 1.5, 1.8, 2.0]  # paper §3.2
M0           = 82 + 60     # tons (shake table + reference bearing weight)

MEASURING_POINTS = [
    "STL_foundation", "STL_2nd_story", "STL_4th_story",
    "CE_1st_story",   "CE_8th_story",  "CE_16th_story",
    "EI_1st_story",   "EI_8th_story",  "EI_15th_story",
]
THRESHOLDS = {
    "STL_foundation": 89, "STL_2nd_story": 89, "STL_4th_story": 89,
    "CE_1st_story":   83, "CE_8th_story":  83, "CE_16th_story": 83,
    "EI_1st_story":   71, "EI_8th_story":  71, "EI_15th_story": 71,
}



In [ ]:
#Start of Actual Program
print("=" * 65)
print("VIBRATION COMFORT ESTIMATION USING LINEAR REGRESSION")
print("=" * 65)

print("\n[1] Loading data from Excel …")
meta_df = pd.read_excel(EXCEL_FILE, sheet_name="Ground_Motion_Metadata")
wal_df  = pd.read_excel(EXCEL_FILE, sheet_name="WAL_Targets")

df = pd.merge(meta_df, wal_df, on="RSN")

VIBRATION COMFORT ESTIMATION USING LINEAR REGRESSION

[1] Loading data from Excel …


In [ ]:
def normalise_signal(signal: np.ndarray, scale_factor: float) -> np.ndarray:
    peak = np.max(np.abs(signal))
    if peak < 1e-10:
        return signal
    return (signal / peak) * scale_factor

#ReSampling Function
def polyphase_resample(signal: np.ndarray, old_fs: float, new_fs: float) -> np.ndarray:
    if old_fs == new_fs:
        return signal
    else:
        count = gcd(old_fs,new_fs)
        down = old_fs / count #Down sampling
        up = new_fs / count #Up Sampling
        resampled = resample_poly(signal, up, down)
        return resampled

#Duration Standardisation
#Target_FS = 50 already defined
def standardise_duration(signal: np.ndarray, fs: int = TARGET_FS, target_dur: float = TARGET_DUR) -> np.ndarray:

    n_target = int(target_dur * fs)
    n        = len(signal)

    if n < n_target:                         # zero-pad
        return np.pad(signal, (0, n_target - n))

    if n > n_target:                         # crop around PGA
        pga_idx = int(np.argmax(np.abs(signal)))
        half    = n_target // 2
        start   = max(0, pga_idx - half)
        end     = start + n_target
        if end > n:
            end   = n
            start = max(0, end - n_target)
        return signal[start:end]

    return signal                            # already 60 s

#Defining the function to calculate MAPE and R2 Score manually
def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

def R2_score(y_true, y_pred):
  SS_res = np.sum((y_true - y_pred)**2)
  SS_tot = np.sum((y_true - np.mean(y_true))**2)
  return 1 - (SS_res / SS_tot)

In [ ]:
def preprocess_record(pga_x: float, pga_y: float, pga_z: float,
                      orig_fs: int, orig_dur: float) -> dict:
    n_orig = int(orig_dur * orig_fs)
    t      = np.arange(n_orig) / orig_fs
    # Simplified waveform: decayed sinusoid + noise
    #Creates a slowly-decaying amplitude envelope.
    #np.exp(-0.08*t) makes the signal die away over time.
    #np.sin(2π×0.15×t) creates a slow modulation that rises and falls and clips negatives so the envelope is always ≥ 0.
    #A real earthquake accelerogram has three physical phases:
    #silence → build-up → strong motion → decay. The envelope simulates this shape
    envelope = np.exp(-0.08 * t) * np.maximum(0, np.sin(2*np.pi*0.15*t))

    # Multiplication by 980.665 converts the PGA from g into gal (cm/s²)
    #This ensures the peak of the synthesised signal matches the PGA value from the dataset.
    # Since our preprocessing only occurs on long time-series data we simulate that by creating
    #this signal from the envelope designed before
    # and a carrier wave which is the actual oscillation: a sinusoid at a characteristic frequency.
    # Noise is also added (instrument noise and small background vibrations
    ax_raw = pga_x * 980.665 * envelope * np.sin(2*np.pi*3.5*t) + np.random.normal(0, 1, n_orig)
    ay_raw = pga_y * 980.665 * envelope * np.sin(2*np.pi*4.0*t + 0.3) + np.random.normal(0, 1, n_orig)
    az_raw = pga_z * 980.665 * envelope * np.sin(2*np.pi*8.0*t + 0.6) + np.random.normal(0, 0.5, n_orig)

    # Step 1 – Normalise
    ax_n = normalise_signal(ax_raw, SCALING[0])
    ay_n = normalise_signal(ay_raw, SCALING[1])
    az_n = normalise_signal(az_raw, SCALING[2])

    # Step 2 – Resample to 50 Hz
    ax_r = polyphase_resample(ax_n, orig_fs, TARGET_FS)
    ay_r = polyphase_resample(ay_n, orig_fs, TARGET_FS)
    az_r = polyphase_resample(az_n, orig_fs, TARGET_FS)

    # Step 3 – Standardise to 60 s
    ax_s = standardise_duration(ax_r)
    ay_s = standardise_duration(ay_r)
    az_s = standardise_duration(az_r)

    return {"ax": ax_s, "ay": ay_s, "az": az_s}

In [ ]:
#Starting the feature coding that defines the dependent and independent variables of the regression
print("\n[3] Engineering features from preprocessed signals …")

#First we preprocess eash record from the raw dataset
records = []
for _, row in df.iterrows():
    proc = preprocess_record(
        pga_x    = row["PGA_X (g)"],
        pga_y    = row["PGA_Y (g)"],
        pga_z    = row["PGA_Z (g)"],
        orig_fs  = int(row["Sampling_Freq (Hz)"]),
        orig_dur = float(row["Duration_Original (s)"]),
    )
    #proc variable now stores the pre-processed rows features for each record and we can extract the inp features from it.
    # Scalar features are extracted
    feat = {
        "RSN"           : int(row["RSN"]), #Identification
        "PGA_norm_X"    : float(row["PGA_X (g)"]), #Normalised Peak Ground Acceleration
        "PGA_norm_Y"    : float(row["PGA_Y (g)"]),
        "PGA_norm_Z"    : float(row["PGA_Z (g)"]),
        "RMS_X"         : float(np.sqrt(np.mean(proc["ax"]**2))), #RMS of time series generated
        "RMS_Y"         : float(np.sqrt(np.mean(proc["ay"]**2))),
        "RMS_Z"         : float(np.sqrt(np.mean(proc["az"]**2))),
        "Arias_X"       : float(np.sum(proc["ax"]**2) / (2 * TARGET_FS)), #This is the total accumulated energy of the signal
        #During dataset generation, the ay and az were generated as a random ratio of ax and hence are linearly depenedent on it.
        #So for linear regression, these features are not included in the feature list as it makes the linear regression split the effect of
        #WAL change on x,y,z values when only x is the independent reading.
        #Removing these two features makes our featutre list smaller and more efficient but the MAPE remains same
        #This is however not the case for real earthquake signals in which all three are indepndent

        # Arias_Y and Arias_Z removed: normalise_signal forces proc["ay"] peak=0.85
        # and proc["az"] peak=0.65 for EVERY record, so Arias_Y ≈ 0.85² × Arias_X
        # and Arias_Z ≈ 0.65² × Arias_X — both are fixed-ratio copies of Arias_X,

        "log_PGA_X"     : float(np.log10(row["PGA_X (g)"] + 1e-6)),

        # log_PGA_Z removed for the same reason: after normalisation the Z waveform
        # peak is always 0.65, so log_PGA_Z tracks log_PGA_X with a constant offset.

        "Magnitude"     : float(row["Magnitude"]), #Magnitude of the earthquake
        "log_Vs30"      : float(np.log10(row["Vs30 (m/s)"])),
        "log_Dist"      : float(np.log10(row["Epicentral_Distance (km)"] + 1)),
    }
    # Append WAL targets
    for mp in MEASURING_POINTS:
        feat[f"WAL_{mp}"] = float(row[f"WAL_{mp}_dB"])
    records.append(feat)

feat_df = pd.DataFrame(records)
print(f"    Feature matrix shape : {feat_df.shape}") #returns the size of the feature matrix

FEATURES = ["PGA_norm_X","PGA_norm_Y","PGA_norm_Z",
            "RMS_X","RMS_Y","RMS_Z",
            "Arias_X",
            "log_PGA_X",
            "Magnitude","log_Vs30","log_Dist"]

X = feat_df[FEATURES].values
#It contains 11 features, 1 RSN, 9 WAL values at different measuring points
print(FEATURES)


[3] Engineering features from preprocessed signals …
    Feature matrix shape : (500, 21)
['PGA_norm_X', 'PGA_norm_Y', 'PGA_norm_Z', 'RMS_X', 'RMS_Y', 'RMS_Z', 'Arias_X', 'log_PGA_X', 'Magnitude', 'log_Vs30', 'log_Dist']


In [ ]:
print("\n[4] Splitting dataset 80 / 20 (train / test) …")
idx_train, idx_test = train_test_split(
    np.arange(len(feat_df)), test_size=0.20, random_state=42
)
X_train, X_test = X[idx_train], X[idx_test]
print(f"    Train samples : {len(idx_train)}   |   Test samples : {len(idx_test)}")


[4] Splitting dataset 80 / 20 (train / test) …
    Train samples : 400   |   Test samples : 100


In [ ]:
print("\n[5] Fitting Linear Regression for each measuring point …\n")
#We run a linear regression model to fit our data to the 9 measuring points

results = {}

for mp in MEASURING_POINTS:
  y = feat_df[f"WAL_{mp}"].values #We extract the values of the WAL of 500 records from the feat_df for WAL_<measuring point the loop is at currently>
  y_train, y_test = y[idx_train], y[idx_test]

  #Now we create the model by instantiating our class
  model = LinearRegression_TRY()
  model.fit(X_train, y_train)
  y_pred_train = model.predict(X_train)
  y_pred_test  = model.predict(X_test)

  mape_train = MAPE(y_train, y_pred_train) * 100
  mape_test  = MAPE(y_test,  y_pred_test)  * 100
  r2_train   = R2_score(y_train, y_pred_train)
  r2_test    = R2_score(y_test,  y_pred_test)

  results[mp] = {
        "model": model,
        "y_train": y_train, "y_pred_train": y_pred_train,
        "y_test":  y_test,  "y_pred_test":  y_pred_test,
        "mape_train": mape_train, "mape_test": mape_test,
        "r2_train": r2_train,     "r2_test":   r2_test,
        "threshold": THRESHOLDS[mp],
    }
  print(f"    {mp:<25} {mape_train:>11.3f}% {mape_test:>11.3f}% {r2_train:>10.4f} {r2_test:>10.4f}")



[5] Fitting Linear Regression for each measuring point …

    STL_foundation                  6.133%       6.429%     0.4022     0.1744
    STL_2nd_story                   6.293%       6.784%     0.3359     0.0600
    STL_4th_story                   6.600%       6.247%     0.3690     0.1912
    CE_1st_story                    6.131%       6.089%     0.3763     0.3214
    CE_8th_story                    6.317%       6.381%     0.3894     0.0788
    CE_16th_story                   6.192%       6.816%     0.3491     0.0644
    EI_1st_story                    6.332%       7.021%     0.3343    -0.0951
    EI_8th_story                    6.271%       6.368%     0.3718     0.0480
    EI_15th_story                   6.362%       5.826%     0.3699     0.1512


In [ ]:
print("\n[7] Generating plots …")

# Scatter Plot: Predicted vs Ground Truth
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
fig.suptitle("Linear Regression: Predicted vs Actual WAL ", fontsize=13, fontweight="bold")

for ax, mp in zip(axes.flat, MEASURING_POINTS):
    r     = results[mp]
    thr   = r["threshold"]
    ymin  = min(r["y_test"].min(),  r["y_pred_test"].min()) - 3
    ymax  = max(r["y_test"].max(),  r["y_pred_test"].max()) + 3
    ax.scatter(r["y_test"], r["y_pred_test"], alpha=0.45, s=18,
               color="#1F6FBF", edgecolors="none", label="Test samples")
    ax.plot([ymin, ymax], [ymin, ymax], "r--", lw=1.5, label="Ideal fit")
    ax.axhline(thr, color="orange", lw=1.2, ls=":", label=f"Threshold {thr} dB")
    ax.axvline(thr, color="orange", lw=1.2, ls=":")
    ax.set_xlim(ymin, ymax); ax.set_ylim(ymin, ymax)
    ax.set_title(mp.replace("_", " "), fontsize=9, fontweight="bold")
    ax.set_xlabel("Ground Truth (dB)", fontsize=8)
    ax.set_ylabel("Prediction (dB)",   fontsize=8)
    ax.annotate(f"R²={r['r2_test']:.3f}\nMAPE={r['mape_test']:.2f}%",
                xy=(0.05, 0.82), xycoords="axes fraction", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="gray"))
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6, loc="lower right")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("scatter_predicted_vs_actual.png", dpi=150, bbox_inches="tight")
plt.close()
print("    Saved → scatter_predicted_vs_actual.png")

#Bar Chart fpr MAPE comparison
fig, ax = plt.subplots(figsize=(12, 5))
x      = np.arange(len(MEASURING_POINTS))
width  = 0.38
mape_tr = [results[mp]["mape_train"] for mp in MEASURING_POINTS]
mape_te = [results[mp]["mape_test"]  for mp in MEASURING_POINTS]
b1 = ax.bar(x - width/2, mape_tr, width, label="Train MAPE (%)", color="#1F6FBF", alpha=0.85)
b2 = ax.bar(x + width/2, mape_te, width, label="Test MAPE (%)",  color="#F4A433", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in MEASURING_POINTS], fontsize=8)
ax.set_ylabel("MAPE (%)", fontsize=10)
ax.set_title("Mean Absolute Percentage Error — Train vs Test\n(Linear Regression surrogate)", fontsize=11)
ax.legend(); ax.grid(axis="y", alpha=0.4)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig("mape_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("    Saved → mape_comparison.png")

# ── 7c  WAL vs PGA (CE 16th story) ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("WAL vs PGA (CE Building – 16th Story)", fontsize=11, fontweight="bold")

mp  = "CE_16th_story"
thr = THRESHOLDS[mp]
for ax, idx, split_lbl in zip(axes, [idx_train, idx_test], ["Train", "Test"]):
    pga  = feat_df["PGA_norm_X"].values[idx]
    wal  = feat_df[f"WAL_{mp}"].values[idx]
    pred = results[mp]["model"].predict(X[idx])
    sc   = ax.scatter(pga, wal,  alpha=0.4, s=15, color="#1F6FBF", label="Actual WAL")
    ax.scatter(pga, pred, alpha=0.4, s=15, color="#E05A2B", marker="x", label="Predicted WAL")
    ax.axhline(thr, color="red", lw=1.5, ls="--", label=f"Threshold {thr} dB")
    ax.set_xlabel("PGA_X (g)", fontsize=9)
    ax.set_ylabel("WAL (dB)",  fontsize=9)
    ax.set_title(f"{split_lbl} Set", fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("wal_vs_pga_CE16.png", dpi=150, bbox_inches="tight")
plt.close()
print("    Saved → wal_vs_pga_CE16.png")

# ── 7d  Preprocessing visualisation for one record ───────────────────────────
np.random.seed(7)
row0   = df.iloc[0]
fs0    = int(row0["Sampling_Freq (Hz)"])
dur0   = float(row0["Duration_Original (s)"])
n_orig = int(dur0 * fs0)
t_orig = np.arange(n_orig) / fs0
envelope = np.exp(-0.08*t_orig)*np.maximum(0, np.sin(2*np.pi*0.15*t_orig))
ax_raw = row0["PGA_X (g)"]*980.665*envelope*np.sin(2*np.pi*3.5*t_orig) + np.random.normal(0,1,n_orig)

ax_n = normalise_signal(ax_raw, SCALING[0])
ax_r = polyphase_resample(ax_n, fs0, TARGET_FS)
ax_s = standardise_duration(ax_r)
t_proc = np.arange(len(ax_s)) / TARGET_FS

fig, axes = plt.subplots(3, 1, figsize=(12, 8))
fig.suptitle(f"Pre-processing Pipeline  |  RSN {int(row0['RSN'])}  "
             f"|  Orig fs={fs0} Hz  |  Orig dur={dur0:.0f} s",
             fontsize=11, fontweight="bold")
axes[0].plot(t_orig, ax_raw, lw=0.6, color="#1F6FBF")
axes[0].set_title("Step 0: Raw X-direction Signal (gal)", fontsize=9)
axes[0].set_ylabel("Acceleration (gal)"); axes[0].grid(alpha=0.3)

axes[1].plot(t_orig, ax_n, lw=0.6, color="#2CA02C")
axes[1].set_title(f"Step 1: Normalised (÷max|a|×{SCALING[0]}) → still at {fs0} Hz", fontsize=9)
axes[1].set_ylabel("Normalised amplitude"); axes[1].grid(alpha=0.3)

axes[2].plot(t_proc, ax_s, lw=0.6, color="#D62728")
axes[2].axvline(60, color="orange", ls="--", lw=1, label="60 s mark")
axes[2].set_title(f"Step 2+3: Resampled to {TARGET_FS} Hz + Standardised to {TARGET_DUR} s", fontsize=9)
axes[2].set_ylabel("Normalised amplitude"); axes[2].set_xlabel("Time (s)")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("preprocessing_pipeline.png", dpi=150, bbox_inches="tight")
plt.close()
print("    Saved → preprocessing_pipeline.png")

print("\n")
print("PIPELINE COMPLETE")



[7] Generating plots …
    Saved → scatter_predicted_vs_actual.png
    Saved → mape_comparison.png
    Saved → wal_vs_pga_CE16.png
    Saved → preprocessing_pipeline.png


PIPELINE COMPLETE
